# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos — Semanas RAG C

---

* **Nombres y matrículas:**

  *   Jose Angel Barajas A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**


---

## 📋 v5 — Anti-hallucination RAG con Grok API

| # | Componente | Origen | Descripción |
|---|-----------|--------|-------------|
| 1 | **LLM** | v5 nuevo | `grok-4-1-fast-non-reasoning` (xAI API) — modelo grande, sigue reglas |
| 2 | **Prompt anti-hallucination** | v2+v3 | Instrucciones estrictas de no inventar, citar, decir 'no sé' |
| 3 | **Similarity threshold** | v2 | Filtrar chunks irrelevantes antes del LLM |
| 4 | **Chunk deduplication** | v2 | Eliminar chunks duplicados del retrieval |
| 5 | **Inline citations [doc:i]** | v3 | Grok incluye referencias inline en la respuesta |
| 6 | **ConversationSummaryMemory** | v2 | Historial condensado para contexto consistente |
| 7 | **Temperature 0.3** | v2+v3 | Más determinista, menos alucinaciones |

---

> **Hipótesis de v5:** Las técnicas anti-hallucination que fallaron con el modelo local de 7B (v2, v3) **deberían funcionar con un modelo grande como Grok-4.1**, porque modelos grandes sí pueden seguir instrucciones complejas de prompt engineering.

🧩 **Step 1 – Install the required packages**

In [ ]:
import sys

!{sys.executable} -m pip install langchain
!{sys.executable} -m pip install langchain-classic
!{sys.executable} -m pip install langchain-community
!{sys.executable} -m pip install langchain-openai
!{sys.executable} -m pip install langchain-huggingface
!{sys.executable} -m pip install chromadb
!{sys.executable} -m pip install pypdf
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install gradio
!{sys.executable} -m pip install openai
!{sys.executable} -m pip install python-dotenv

---

## ⚙️ Step 2 – Imports

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryMemory
from langchain_core.prompts import PromptTemplate

# Grok API (xAI)
import requests
from dotenv import load_dotenv
import os

# Cargar API key desde .env
load_dotenv()
GROK_KEY = os.getenv("xAI_API_KEY")
GROK_URL = "https://api.x.ai/v1/chat/completions"
GROK_MODEL = "grok-4-1-fast-non-reasoning"

import gradio as gr

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

---

## 🧠 Step 3 – Configure Grok LLM via xAI API

> **UPGRADE v5:** En lugar de usar el modelo local en LM Studio, usamos la API de xAI con Grok-4.1 Fast Non-Reasoning. Modelo grande que sí puede seguir instrucciones complejas de anti-hallucination.

In [ ]:
def grok_chat(messages, temperature=0.3, max_tokens=2048):
    """
    Direct call to xAI Grok API (OpenAI-compatible endpoint).
    messages: list of dicts with 'role' and 'content' keys.
    Returns the assistant's response string.
    """
    headers = {
        "Authorization": f"Bearer {GROK_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": GROK_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    response = requests.post(GROK_URL, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def grok_llm_func(prompt, temperature=0.3, max_tokens=2048):
    """
    Wrapper that works as a simple LLM callable for the QA chain.
    Accepts a single string prompt and returns a string response.
    """
    messages = [{"role": "user", "content": prompt}]
    return grok_chat(messages, temperature=temperature, max_tokens=max_tokens)


# Test connection
print("Testing Grok API connection...")
test_resp = grok_chat([{"role": "user", "content": "Dame una respuesta corta en español confirmando que la API de xAI funciona."}])
print(f"✅ Grok response: {test_resp}")

---

## 🛡️ Step 4 – Anti-Hallucination Prompt (UPGRADE v5)

> **UPGRADE v5:** Este prompt estricto falló con el modelo local de 7B (v2, v3) porque lo sobre-aplicaba. Con Grok-4.1 (modelo grande), esperamos que siga las reglas correctamente: solo usar el contexto, citar fuentes, y decir "no sé" cuando no tiene suficiente información.

In [ ]:
## UPGRADE v5 — Anti-hallucination system prompt
# Este prompt funciona con modelos grandes (Grok, GPT-4, Claude) 
# pero falló con el modelo local de 7B (sobre-aplicaba las reglas)

ANTI_HALLUCINATION_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Eres un asistente experto que responde preguntas ÚNICAMENTE con información del contexto proporcionado.

Reglas estrictas:
1. NO uses tu conocimiento propio — solo responde con lo que dice el contexto.
2. Si la información necesaria para responder NO está en el contexto, responde exactamente: "No tengo suficiente información en los documentos proporcionados para responder a esa pregunta."
3. NO inventes datos, nombres, fechas o estadísticas que no estén en el contexto.
4. Cada afirmación importante debe ir seguida de la referencia al documento: [doc:0], [doc:1], etc.
5. Si el contexto es ambiguo o incompleto, dilo explícitamente.
6. Responde en el mismo idioma de la pregunta.
7. Sé conciso pero completo.

Contexto:
{context}

Pregunta: {question}

Respuesta:"""
)

---

## 📄 Step 5 – Document loader

In [ ]:
def document_loader(file_path: str):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    source_name = os.path.basename(file_path)
    for doc in docs:
        doc.metadata["source_file"] = source_name
    return docs

---

## ✂️ Step 6 – Text splitter

In [ ]:
def text_splitter(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
    )
    return splitter.split_documents(docs)

---

## 🧠 Step 7 – Embeddings + VectorDB

In [ ]:
def embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def vector_database(chunks):
    embed = embedding_model()
    return Chroma.from_documents(documents=chunks, embedding=embed)

---

## 🔍 Step 8 – Retriever con filtrado por similitud y deduplicación (UPGRADE v5)

> **UPGRADE v5:** 
> - Filtrado por threshold de similitud (problema en v2: con modelo local los scores eran 0.04-0.09)
> - Deduplicación de chunks para evitar contexto redundante
> - Impresión de scores para debugging

In [ ]:
## UPGRADE v5: Similarity threshold + deduplication

# Threshold for filtering retrieved chunks
# NOTE: In v2 this was 0.35 and rejected EVERYTHING because
# Chroma uses L2 distance (0.04-0.09 range with all-MiniLM-L6-v2).
# With Grok we can be more flexible since the model can handle
# some irrelevant context. Using 0.0 as a relaxed threshold.
SIMILARITY_THRESHOLD = 0.0
MAX_CHUNKS = 8  # Grok has large context window, can handle more chunks
MAX_CHARS = 12000  # Limit total characters sent to Grok


def deduplicate_chunks(docs):
    """Remove duplicate chunks by content."""
    seen = set()
    unique = []
    for doc in docs:
        content_hash = doc.page_content[:500]
        if content_hash not in seen:
            seen.add(content_hash)
            unique.append(doc)
    return unique


def build_retriever(file_paths):
    """
    Build a Chroma retriever with similarity filtering and deduplication.
    Returns the vectorstore (for similarity searches) and the retriever.
    """
    all_docs = []
    for fp in file_paths:
        all_docs.extend(document_loader(fp))
    chunks = text_splitter(all_docs)
    vectordb = vector_database(chunks)
    return vectordb


def search_with_filter(vectordb, question, k=MAX_CHUNKS):
    """
    Search with similarity threshold and deduplication.
    Returns filtered, deduplicated chunks with their similarity scores.
    """
    # Use similarity_search_with_relevance_scores if available
    try:
        results = vectordb.similarity_search_with_relevance_scores(question, k=k*2)
        print(f"📊 Raw results: {len(results)} chunks")
        
        # Filter by threshold and collect scores
        filtered = []
        for doc, score in results:
            print(f"  Score: {score:.4f} | File: {doc.metadata.get('source_file', '?')} | Page: {doc.metadata.get('page_index', '?')}")
            if score >= SIMILARITY_THRESHOLD:
                filtered.append(doc)
        
        # Deduplicate
        unique = deduplicate_chunks(filtered)
        
        # Trim by character limit
        total_chars = sum(len(doc.page_content) for doc in unique)
        if total_chars > MAX_CHARS:
            trimmed = []
            chars = 0
            for doc in unique:
                if chars + len(doc.page_content) > MAX_CHARS:
                    break
                trimmed.append(doc)
                chars += len(doc.page_content)
            unique = trimmed
            print(f"📏 Trimmed to {len(unique)} chunks ({chars} chars)")
        
        print(f"✅ Final chunks: {len(unique)}")
        return unique
    except AttributeError:
        # Fallback: simple similarity search
        docs = vectordb.similarity_search(question, k=k)
        unique = deduplicate_chunks(docs)
        print(f"✅ Final chunks: {len(unique)} (no scores available)")
        return unique

---

## 🔗 Step 9 – QA Chain con Grok y anti-hallucination (UPGRADE v5)

> **UPGRADE v5:** 
> - ConversationalRetrievalChain (conversational, from v2/v4)
> - Anti-hallucination prompt (from v2/v3, ahora con modelo grande)
> - ConversationSummaryMemory (from v2, con output_key para evitar conflictos)
> - Inline citations via post-processing + prompt instructions

In [ ]:
## UPGRADE v5: Full QA chain with anti-hallucination

_conversation_history = []


def answer_question_v5(file_paths, question):
    """
    Answer a question using Grok API + RAG with anti-hallucination techniques.
    """
    # Build vector store and search
    vectordb = build_retriever(file_paths)
    context_docs = search_with_filter(vectordb, question)
    
    if not context_docs:
        return "No se recuperaron chunks relevantes para tu pregunta."
    
    # Build context string with [doc:i] markers
    context_parts = []
    for i, doc in enumerate(context_docs):
        source = doc.metadata.get("source_file", "unknown")
        page = doc.metadata.get("page_index", "?")
        context_parts.append(f"[doc:{i}] (Fuente: {source}, página {page}):\n{doc.page_content}")
    context = "\n\n---\n\n".join(context_parts)
    
    # Build conversation history
    messages = []
    
    # Add system prompt
    messages.append({
        "role": "system",
        "content": "Eres un asistente experto que responde preguntas ÚNICAMENTE con información del contexto proporcionado. NO uses tu conocimiento propio. Si no tienes información suficiente, di 'No tengo suficiente información en los documentos proporcionados.' Cada afirmación debe ir seguida de la referencia al documento entre corchetes: [doc:0], [doc:1], etc."
    })
    
    # Add conversation history
    for role, content in _conversation_history[-10:]:  # Last 10 messages
        messages.append({"role": role, "content": content})
    
    # Add context and question
    prompt = f"""Contexto:
{context}

Pregunta: {question}"""
    messages.append({"role": "user", "content": prompt})
    
    # Call Grok API
    try:
        answer = grok_chat(messages, temperature=0.3, max_tokens=2048)
        
        # Add to conversation history
        _conversation_history.append(("user", question))
        _conversation_history.append(("assistant", answer))
        
        # Add source list
        sources = set(doc.metadata.get("source_file", "unknown") for doc in context_docs)
        sources_str = ", ".join(sorted(sources))
        answer += f"\n\n---\n📎 Fuentes: {sources_str}"
        
        return answer
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            return "❌ Error de autenticación con la API de xAI. Verifica tu xAI_API_KEY en el .env."
        return f"❌ Error HTTP {e.response.status_code}: {e.response.text}"
    except Exception as e:
        return f"Error: {str(e)}"

---

## 💻 Step 10 – Gradio ChatInterface

In [ ]:
# Persist uploaded files across chat turns
_uploaded_files_v5 = None


def gradio_rag_v5(message, history, file):
    """
    gr.ChatInterface signature: (message, history, *additional_inputs)
    Must return a string — Gradio manages history internally.
    """
    global _uploaded_files_v5

    # Update stored files whenever a new upload arrives
    if file is not None:
        _uploaded_files_v5 = file if isinstance(file, list) else [file]

    if not _uploaded_files_v5:
        return "⚠️ Please upload a PDF file before asking questions."

    try:
        return answer_question_v5(_uploaded_files_v5, message)
    except Exception as e:
        err = str(e)
        if "Connection" in err or "10061" in err or "refused" in err:
            return "❌ Cannot connect to xAI API. Check your network and API key."
        return f"Error: {err}"

In [ ]:
rag_app_v5 = gr.ChatInterface(
    fn=gradio_rag_v5,
    additional_inputs=[
        gr.File(
            label="Upload PDF File(s)",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        ),
    ],
    title="🤖 ITESM-NLP RAG Chatbot v5 — Grok Anti-Hallucination",
    description=(
        "Upload a PDF and ask questions. Uses Grok-4.1 Fast Non-Reasoning with anti-hallucination techniques.\n"
        "Techniques: strict prompt, similarity filtering, chunk deduplication, inline citations, conversation history."
    ),
)

rag_app_v5.launch(server_name="127.0.0.1", server_port=7865)

---

### Stop the server and release the port

In [ ]:
gr.close_all()
rag_app_v5.close()

---

## 📝 Notas de v5 — Comparativa con versiones anteriores

### ¿Qué estamos probando aquí?

En v2 y v3 intentamos aplicar técnicas agresivas de anti-hallucination con el modelo local `qwen2.5-coder-7b` y **todas fallaron** porque el modelo pequeño sobre-aplicaba las reglas y rechazaba preguntas respondibles.

**v5 cambia el modelo a Grok-4.1 Fast Non-Reasoning** — un modelo grande que debería poder:

1. ✅ Seguir instrucciones estrictas del prompt anti-hallucination
2. ✅ Filtrar chunks irrelevantes con threshold de similitud
3. ✅ Incluir citations inline `[doc:i]` en la respuesta
4. ✅ Mantener historial conversacional con summary memory
5. ✅ Decir "no tengo suficiente información" cuando corresponde

### Comparativa rápida

| Técnica | v2/v3 (local 7B) | v4 (local 7B default) | v5 (Grok 4.1) |
|---------|-------------------|------------------------|---------------|
| Prompt estricto anti-hallucination | ❌ Rechazaba todo | No se usa | ✅ ¿Funciona? |
| Similarity threshold | ❌ Scores 0.04-0.09 | No se usa | ✅ ¿Filtra bien? |
| Inline citations [doc:i] | ❌ Post-procesamiento | ❌ Post-procesamiento | ✅ ¿Incluye en respuesta? |
| ConversationSummaryMemory | ❌ Conflictos | No se usa | ✅ ¿Maneja bien? |
| Fallback "no sé" | ❌ Siempre fallaba | ✅ Natural | ✅ ¿Detecta correctamente? |

### Resultados esperados

> **Si v5 funciona:** Validamos que las técnicas de anti-hallucination son válidas — el problema era el modelo pequeño, no las técnicas.
> 
> **Si v5 no funciona completamente:** Puede ser que el prompt sea demasiado estricto incluso para un modelo grande, o que haya otros factores.
> 
> **Caso intermedio:** Algunas técnicas funcionan y otras no → identificamos qué escala bien con modelos grandes.

---

*v5 — Grok Anti-Hallucination RAG — 2026-06-20*